# Ejercicio 1 — Un ejemplo pequeño para razonar

```
R1:  Llueve ⇒ Mojada
R2:  Riego ⇒ Mojada
R3:  Mojada
```

Primero se razona a mano; el código de cada punto sirve para verificar.

In [1]:
from sympy import symbols, And, Or, Not, Implies, Equivalent
from sympy.logic.boolalg import truth_table
from sympy.logic.inference import satisfiable


def consultar(base, proposicion):
    if satisfiable(base) is False:
        return "Base inconsistente"
    if satisfiable(And(base, Not(proposicion))) is False:
        return "Verdadero"
    if satisfiable(And(base, proposicion)) is False:
        return "Falso"
    return "Desconocido"


Llueve, Riego, Mojada = symbols("Llueve Riego Mojada")

R1 = Implies(Llueve, Mojada)
R2 = Implies(Riego, Mojada)
R3 = Mojada
base = And(R1, R2, R3)

## ¿Cuántos modelos posibles hay? ¿En cuántos de ellos es verdadera la base?

A mano: tres símbolos, cada uno verdadero o falso: 2 × 2 × 2 = **8 modelos
posibles**. `R3` obliga a que `Mojada` sea verdadera, lo que descarta 4. En los
4 que quedan `R1` y `R2` se cumplen solas, porque una implicación con
consecuente verdadero es verdadera. **La base es verdadera en 4.**

In [2]:
total = 0
verdaderos = 0
for valores, resultado in truth_table(base, [Llueve, Riego, Mojada]):
    print(valores, resultado)
    total = total + 1
    if resultado:
        verdaderos = verdaderos + 1

print("modelos posibles:", total)
print("modelos de la base:", verdaderos)

[0, 0, 0] False
[0, 0, 1] True
[0, 1, 0] False
[0, 1, 1] True
[1, 0, 0] False
[1, 0, 1] True
[1, 1, 0] False
[1, 1, 1] True
modelos posibles: 8
modelos de la base: 4


## ¿Se sigue `Llueve`? ¿Se sigue `¬Llueve`?

Una sentencia se sigue de la base cuando es verdadera en **todos** los modelos
de la base. En los 4 modelos de arriba `Llueve` vale 1 en dos y 0 en los otros
dos: no se sigue `Llueve` y tampoco se sigue `¬Llueve`. `consultar` responde
**Desconocido** a las dos.

No se contradicen: "no puedo demostrar que llueve" y "no puedo demostrar que no
llueve" son compatibles. La base simplemente no dice nada de la lluvia. De "si
llueve, se moja" y "está mojada" no sale "llueve": el piso pudo mojarse por el
riego.

In [3]:
print("Llueve: ", consultar(base, Llueve))
print("¬Llueve:", consultar(base, Not(Llueve)))

Llueve:  Desconocido
¬Llueve: Desconocido


## Agregá `R4: ¬Riego`

Los modelos de la base bajan a 2: `Mojada` verdadera, `Riego` falso, `Llueve`
libre. Las respuestas **no cambian**: `Llueve` y `¬Llueve` siguen Desconocido.

Descartamos el riego, pero `R1` y `R2` solo dicen qué pasa *si* llueve o *si*
se riega. Ninguna regla dice que estar mojada obligue a alguna causa. El modelo
"no llueve, no se riega, está mojada" sigue siendo válido: la base no excluye
una tercera causa que no nombramos.

In [4]:
R4 = Not(Riego)
base = And(R1, R2, R3, R4)

for valores, resultado in truth_table(base, [Llueve, Riego, Mojada]):
    if resultado:
        print(valores)

print("Llueve: ", consultar(base, Llueve))
print("¬Llueve:", consultar(base, Not(Llueve)))

[0, 0, 1]
[1, 0, 1]
Llueve:  Desconocido
¬Llueve: Desconocido


## Reemplazá `R1` y `R2` por `Mojada ⇔ (Llueve ∨ Riego)`

Queda **un solo modelo**: llueve, no se riega, está mojada. Ahora `Llueve` es
**Verdadero** y `¬Llueve` es **Falso**.

"Si llueve, se moja" es una sola dirección: `Llueve ⇒ Mojada`. "Se moja
exactamente cuando llueve o se riega" es un bicondicional que agrega la
dirección contraria, `Mojada ⇒ (Llueve ∨ Riego)`: si está mojada, alguna de
las dos causas tiene que valer, y no hay otra. Esa dirección es la que permite
razonar del efecto a la causa: con el riego descartado, solo queda la lluvia.

In [5]:
R5 = Equivalent(Mojada, Or(Llueve, Riego))
base = And(R5, R3, R4)

for valores, resultado in truth_table(base, [Llueve, Riego, Mojada]):
    if resultado:
        print(valores)

print("Llueve: ", consultar(base, Llueve))
print("¬Llueve:", consultar(base, Not(Llueve)))

[1, 0, 1]
Llueve:  Verdadero
¬Llueve: Falso
